# VLJEPA

In [ ]:
#
import json

with open(r"C:\Users\louai\Net2Brain\notebooks\VLJEPA\LeVLJEPA\config.json", "r") as f:
    config = json.load(f)

print(config)

In [ ]:
#
from safetensors.torch import load_file

state_dict = load_file(
    r"C:\Users\louai\Net2Brain\notebooks\VLJEPA\LeVLJEPA\vision_encoder.safetensors"
)

print(f"Number of keys: {len(state_dict)}")

for i, key in enumerate(state_dict.keys()):
    print(key)
    if i == 30:
        break

In [ ]:
#
from safetensors.torch import load_file

state_dict = load_file(
    r"C:\Users\louai\Net2Brain\notebooks\VLJEPA\LeVLJEPA\vision_encoder.safetensors"
)

for key in ["pre_proj.0.weight", "pre_proj.3.weight"]:
    print(key, state_dict[key].shape)

In [ ]:
#
from safetensors.torch import load_file

vision_weights = load_file(
    r"C:\Users\louai\Net2Brain\notebooks\VLJEPA\LeVLJEPA\vision_encoder.safetensors"
)

missing, unexpected = model.load_state_dict(
    {k[len("encoder."):]: v
     for k, v in vision_weights.items()
     if k.startswith("encoder.")},
    strict=False
)

print("Missing:", missing)
print("Unexpected:", unexpected)

In [ ]:
import timm
import torch
from safetensors.torch import load_file

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create the backbone
levljepa_model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=0,
).to(device)

# Load LeVLJEPA checkpoint
vision_weights = load_file(
    r"C:\Users\louai\Net2Brain\notebooks\VLJEPA\LeVLJEPA\vision_encoder.safetensors"
)

missing, unexpected = levljepa_model.load_state_dict(
    {
        k[len("encoder."):]: v
        for k, v in vision_weights.items()
        if k.startswith("encoder.")
    },
    strict=False
)

print("Missing:", missing)
print("Unexpected:", unexpected)

levljepa_model.eval()

# INFONCE

In [ ]:
#
from huggingface_hub import hf_hub_download

config_path = hf_hub_download(
    repo_id="lukaskuhndkfz/InfoNCE-ViT-B-DataComp-200k",
    filename="config.json"
)

print(config_path)

In [ ]:
import json

with open(config_path, "r") as f:
    config = json.load(f)

print(config)

In [ ]:
import timm

model_infonce = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=0,
)

#print(type(model_infonce))
#print(model_infonce)

In [ ]:
# checkpoint loading into model
from safetensors.torch import load_file

vision_weights = load_file(
    hf_hub_download(
        "lukaskuhndkfz/InfoNCE-ViT-B-DataComp-200k",
        "vision_encoder.safetensors"
    )
)

missing, unexpected = model_infonce.load_state_dict(
    {k[len("encoder."):]: v
     for k, v in vision_weights.items()
     if k.startswith("encoder.")},
    strict=False
)

print("Missing:", missing)
print("Unexpected:", unexpected)

In [ ]:
print(hasattr(model_infonce, "prepare_tokens"))

In [ ]:
print(model_infonce.forward_features)

In [ ]:
import inspect

print(inspect.getsource(model_infonce.forward_features))

# CLIP

# feat extractor

In [ ]:
from net2brain.utils.download_datasets import DatasetBonnerPNAS2017, DatasetAlgonauts_NSD_Shared
from pprint import pprint
import os

#paths_bonner = DatasetBonnerPNAS2017.load_dataset()
#pprint(paths_bonner)

paths_NSD_shared  = DatasetAlgonauts_NSD_Shared.load_dataset()
#pprint(paths_NSD_shared)
#paths_NSD_shared.keys()


stimuli_path = paths_NSD_shared["images"]

# count files
# num_images = len([f for f in os.listdir(stimuli_path) if f.endswith(('.jpg', '.png'))])

#print("Number of images:", num_images)

In [ ]:
from torchvision import transforms as T
from torchvision import transforms as trn
import torchextractor as tx
from net2brain.feature_extraction import FeatureExtractor

def my_preprocessor(image, model_name, device):
    """
    Args:
        image (Union[Image.Image, List[Image.Image]]): A PIL Image or a list of PIL Images.
        model_name (str): The name of the model, used to determine specific preprocessing if necessary.
        device (str): The device to which the tensor should be transferred ('cuda' for GPU, 'cpu' for CPU).

    Returns:
        Union[torch.Tensor, List[torch.Tensor]]: The preprocessed image(s) as PyTorch tensor(s).
    """

    transforms = trn.Compose([
        trn.Resize((224, 224)),
        trn.ToTensor(),
        trn.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    img_tensor = transforms(image).unsqueeze(0)
    if device == 'cuda':
        img_tensor = img_tensor.cuda()

    return img_tensor

def my_infonce_transformer_extractor(
        preprocessed_data,
        layers_to_extract,
        model,
        pooling="patch_mean"  # can be changed to pooling="patch_mean" 
):

    vit = model.model

    features = {}

    with torch.no_grad():

        x = vit.patch_embed(preprocessed_data)

        cls = vit.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls, x), dim=1)

        x = vit.pos_drop(x + vit.pos_embed)

        for i, block in enumerate(vit.blocks):
            x = block(x)
    
            if pooling == "cls":
                feat = x[:, 0]
            else:
                feat = x[:, 1:, :].mean(dim=1)

            features[f"layer_{i+1}"] = feat

    return features

def my_cleaner(features):
    return features

In [ ]:
import torch.nn as nn

class InfoNCEFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

infonce_wrapper = InfoNCEFeatureExtractor(model_infonce)
levljepa_wrapper = InfoNCEFeatureExtractor(levljepa_model)

In [ ]:
from net2brain.feature_extraction import FeatureExtractor
import torch
from PIL import Image
import os



fx = FeatureExtractor(
    model=infonce_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_infonce_transformer_extractor
)

fx.extract(
    data_path=stimuli_path,
    save_path=r'C:\Users\louai\Net2Brain\notebooks\0_FEATURE EXTRACTIONS\INFONCE_Feat_nsd_mean'
)

In [ ]:
from net2brain.feature_extraction import FeatureExtractor
import torch
fx = FeatureExtractor(
    model=levljepa_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_infonce_transformer_extractor
)

fx.extract(
    data_path=stimuli_path,
    save_path=r'C:\Users\louai\Net2Brain\notebooks\0_FEATURE EXTRACTIONS\VLJEPA_Feat_nsd_mean'
)

In [ ]:
#
import inspect

print(inspect.getsource(model.forward))